In [ ]:
import os
import sys
import subprocess
import json
import requests

from google.colab import drive, userdata
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

REPO_DIR = "/content/teachmewrite"
REPO_URL = "https://github.com/edmedpublic-hub/teachmewrite.git"

if os.path.exists(REPO_DIR):
    print("Repo exists. Pulling latest changes...")
    subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", "main"], capture_output=True)
else:
    print("Cloning repo...")
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

print("Installing dependencies...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "requests", "anthropic"], check=True)

try:
    os.environ["ANTHROPIC_API_KEY"] = userdata.get('ANTHROPIC_API_KEY')
    print("Secrets loaded successfully.")
except Exception as e:
    print(f"ERROR loading secrets: {e}")
    raise

LIVE_URL = "https://teachmewrite-ai-195673983724.asia-south1.run.app"
NOTES_DB = "/content/drive/MyDrive/teachmewrite-ai/student_notes.json"

# Initialize notes DB if it doesn't exist
if not os.path.exists(NOTES_DB):
    with open(NOTES_DB, "w") as f:
        json.dump({}, f)
    print("Initialized empty notes DB.")

assert os.environ.get("ANTHROPIC_API_KEY") is not None, "ANTHROPIC_API_KEY is missing!"
print(f"Target Live Service: {LIVE_URL}")
print(f"Notes DB: {NOTES_DB}")
print("✅ Cell 1a complete. Environment is ready.")

Mounted at /content/drive
Cloning repo...
Installing dependencies...
Secrets loaded successfully.
Initialized empty notes DB.
Target Live Service: https://teachmewrite-ai-195673983724.asia-south1.run.app
Notes DB: /content/drive/MyDrive/teachmewrite-ai/student_notes.json
✅ Cell 1a complete. Environment is ready.


In [ ]:
from anthropic import Anthropic

client = Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])

# --- Tool Schemas ---
tools = [
    {
        "name": "get_writing_feedback",
        "description": "Get structured writing feedback (grammar issues, clarity problems). Use when the student wants to understand their mistakes.",
        "input_schema": {
            "type": "object",
            "properties": {
                "student_text": {"type": "string"},
                "level": {"type": "integer"}
            },
            "required": ["student_text", "level"]
        }
    },
    {
        "name": "get_rewrite_suggestion",
        "description": "Get ONLY the corrected version of the text. Use when the student says 'just fix it for me'.",
        "input_schema": {
            "type": "object",
            "properties": {
                "student_text": {"type": "string"},
                "level": {"type": "integer"}
            },
            "required": ["student_text", "level"]
        }
    },
    {
        "name": "save_student_note",
        "description": "Save a short note about the student's progress, mistakes, or what was worked on. Call this after giving feedback or a rewrite so you remember the student's context for future turns.",
        "input_schema": {
            "type": "object",
            "properties": {
                "student_id": {"type": "string", "description": "Unique identifier for the student, e.g., 'ahmed'."},
                "note": {"type": "string", "description": "A short 1-2 sentence note, e.g., 'Struggles with subject-verb agreement (I am agree).'"}
            },
            "required": ["student_id", "note"]
        }
    },
    {
        "name": "recall_student_notes",
        "description": "Retrieve all previously saved notes about a student. Use when the student asks 'what did we work on', 'what do you remember about me', or you need past context.",
        "input_schema": {
            "type": "object",
            "properties": {
                "student_id": {"type": "string", "description": "Unique identifier for the student."}
            },
            "required": ["student_id"]
        }
    }
]

# --- Tool Execution Functions ---
def get_writing_feedback(student_text: str, level: int) -> dict:
    print(f"  [Tool] get_writing_feedback (level {level})")
    try:
        r = requests.post(f"{LIVE_URL}/feedback",
                          json={"student_text": student_text, "level": level}, timeout=45)
        r.raise_for_status()
        return r.json()
    except Exception as e:
        return {"error": str(e)}

def get_rewrite_suggestion(student_text: str, level: int) -> dict:
    print(f"  [Tool] get_rewrite_suggestion (level {level})")
    try:
        r = requests.post(f"{LIVE_URL}/feedback",
                          json={"student_text": student_text, "level": level}, timeout=45)
        r.raise_for_status()
        return {"rewrite_suggestion": r.json().get("rewrite_suggestion", "")}
    except Exception as e:
        return {"error": str(e)}

def save_student_note(student_id: str, note: str) -> dict:
    print(f"  [Tool] save_student_note (student: {student_id})")
    try:
        with open(NOTES_DB, "r") as f:
            db = json.load(f)
        db.setdefault(student_id, []).append(note)
        with open(NOTES_DB, "w") as f:
            json.dump(db, f, indent=2)
        return {"status": "saved", "note_count": len(db[student_id])}
    except Exception as e:
        return {"error": str(e)}

def recall_student_notes(student_id: str) -> dict:
    print(f"  [Tool] recall_student_notes (student: {student_id})")
    try:
        with open(NOTES_DB, "r") as f:
            db = json.load(f)
        notes = db.get(student_id, [])
        return {"student_id": student_id, "notes": notes, "count": len(notes)}
    except Exception as e:
        return {"error": str(e)}

TOOL_FUNCTIONS = {
    "get_writing_feedback": get_writing_feedback,
    "get_rewrite_suggestion": get_rewrite_suggestion,
    "save_student_note": save_student_note,
    "recall_student_notes": recall_student_notes,
}

print(f"✅ Cell 1b complete. {len(tools)} tools defined: {list(TOOL_FUNCTIONS.keys())}")

✅ Cell 1b complete. 4 tools defined: ['get_writing_feedback', 'get_rewrite_suggestion', 'save_student_note', 'recall_student_notes']


In [ ]:
class StatefulTutorAgent:
    def __init__(self, student_id: str, student_level: int = 5):
        self.student_id = student_id
        self.student_level = student_level
        self.conversation_history = []  # persists across turns
        self.system_prompt = f"""You are a stateful English writing tutor for a Pakistani student (ID: {student_id}, Level: {student_level}).

You have FOUR tools:
1. `get_writing_feedback` — structured feedback (for "help me understand")
2. `get_rewrite_suggestion` — corrected text (for "just fix it")
3. `save_student_note` — save what you worked on, so you remember for next time
4. `recall_student_notes` — recall past notes about this student

Behavioral rules:
- After giving feedback OR a rewrite, ALWAYS call `save_student_note` to record what you worked on. You may call multiple tools in one turn.
- If the student asks "what did we work on" or "what do you remember", call `recall_student_notes`.
- Be brief, encouraging, and Socratic when explaining.
- Always use student_id = "{student_id}" for save/recall calls.
"""

    def chat(self, user_message: str, max_iterations: int = 6) -> dict:
        # Append user message to persistent history
        self.conversation_history.append({"role": "user", "content": user_message})

        tools_called_this_turn = []

        for iteration in range(max_iterations):
            print(f"  [Iteration {iteration + 1}]")
            response = client.messages.create(
                model="claude-haiku-4-5-20251001",
                max_tokens=1024,
                system=self.system_prompt,
                tools=tools,
                messages=self.conversation_history
            )

            if response.stop_reason == "tool_use":
                # Add Claude's assistant turn (may contain multiple tool_use blocks)
                self.conversation_history.append({"role": "assistant", "content": response.content})

                # Execute ALL tool calls in this turn
                tool_results = []
                for block in response.content:
                    if block.type == "tool_use":
                        tool_name = block.name
                        tools_called_this_turn.append(tool_name)
                        print(f"  [Agent] Tool: {tool_name}")

                        if tool_name in TOOL_FUNCTIONS:
                            result = TOOL_FUNCTIONS[tool_name](**block.input)
                        else:
                            result = {"error": f"Unknown tool: {tool_name}"}

                        tool_results.append({
                            "type": "tool_result",
                            "tool_use_id": block.id,
                            "content": json.dumps(result)
                        })

                # Append all tool results as a single user message
                self.conversation_history.append({"role": "user", "content": tool_results})

            elif response.stop_reason == "end_turn":
                final_text = "".join(b.text for b in response.content if hasattr(b, "text"))
                # Append assistant's final answer to history
                self.conversation_history.append({"role": "assistant", "content": final_text})
                return {"response": final_text, "tools_called": tools_called_this_turn}

            else:
                return {"response": f"Unexpected stop_reason: {response.stop_reason}", "tools_called": tools_called_this_turn}

        return {"response": "Max iterations reached.", "tools_called": tools_called_this_turn}

print("✅ Cell 1c complete. StatefulTutorAgent is ready.")

✅ Cell 1c complete. StatefulTutorAgent is ready.


In [ ]:
student_text = "I am agree with my father that education is more better than money."
student_id = "ahmed_test_93"

# Reset notes DB for this student to start clean
with open(NOTES_DB, "r") as f:
    db = json.load(f)
db[student_id] = []
with open(NOTES_DB, "w") as f:
    json.dump(db, f, indent=2)

agent = StatefulTutorAgent(student_id=student_id, student_level=5)

# --- Turn 1: Fix it ---
print("=" * 60)
print("TURN 1: 'Fix my essay.'")
print("=" * 60)
t1 = agent.chat(f"{student_text} — Fix it for me.")
print(f"\nTools: {t1['tools_called']}")
print(f"Agent: {t1['response']}\n")

# --- Turn 2: Explain ---
print("=" * 60)
print("TURN 2: 'Now explain what I got wrong.'")
print("=" * 60)
t2 = agent.chat("Now explain what I got wrong.")
print(f"\nTools: {t2['tools_called']}")
print(f"Agent: {t2['response']}\n")

# --- Turn 3: Recall ---
print("=" * 60)
print("TURN 3: 'What did we work on today?'")
print("=" * 60)
t3 = agent.chat("What did we work on today?")
print(f"\nTools: {t3['tools_called']}")
print(f"Agent: {t3['response']}\n")

# --- Verification ---
print("=" * 60)
print("VERIFICATION")
print("=" * 60)
assert "save_student_note" in t1["tools_called"], f"Turn 1 should save a note, got: {t1['tools_called']}"
assert "get_writing_feedback" in t2["tools_called"], f"Turn 2 should call feedback, got: {t2['tools_called']}"
assert "recall_student_notes" in t3["tools_called"], f"Turn 3 should recall notes, got: {t3['tools_called']}"
print("✅ All 3 turns routed correctly.")

# Check that Turn 3 referenced Turn 1 context
t3_lower = t3["response"].lower()
assert any(k in t3_lower for k in ["agree", "better", "grammar"]), \
    f"Turn 3 should reference Turn 1 material. Got: {t3['response']}"
print("✅ Turn 3 recalled prior context.")
print("✅ Cell 1d complete. Stateful agent verified.")


TURN 1: 'Fix my essay.'
  [Iteration 1]
  [Agent] Tool: get_rewrite_suggestion
  [Tool] get_rewrite_suggestion (level 5)
  [Agent] Tool: save_student_note
  [Tool] save_student_note (student: ahmed_test_93)
  [Iteration 2]

Tools: ['get_rewrite_suggestion', 'save_student_note']
Agent: **Here's the corrected version:**

> I agree with my father that education is better than money.

**Two quick fixes:**
1. **"I am agree"** → **"I agree"** (no helping verb needed)
2. **"more better"** → **"better"** (don't use both "more" and the "-er" ending together)

Well done on your thought! 👍

TURN 2: 'Now explain what I got wrong.'
  [Iteration 1]
  [Agent] Tool: get_writing_feedback
  [Tool] get_writing_feedback (level 5)
  [Iteration 2]

Tools: ['get_writing_feedback']
Agent: **Here's what went wrong:**

1. **"I am agree"** (High severity)
   - The verb *agree* doesn't pair with "am." 
   - Think of it this way: "am" is for describing states (*I am happy, I am here*), but "agree" is an action ver

In [ ]:
agent_artifact = {
    "student_id": student_id,
    "student_text": student_text,
    "conversation": [
        {"turn": 1, "user": "Fix it for me.", "tools": t1["tools_called"], "response": t1["response"]},
        {"turn": 2, "user": "Now explain what I got wrong.", "tools": t2["tools_called"], "response": t2["response"]},
        {"turn": 3, "user": "What did we work on today?", "tools": t3["tools_called"], "response": t3["response"]}
    ],
    "persistent_state": json.load(open(NOTES_DB))[student_id],
    "verification": "3-turn stateful conversation passed all assertions",
    "model": "claude-haiku-4-5-20251001"
}

save_path = "/content/drive/MyDrive/teachmewrite-ai/agent_stateful_day93.json"
with open(save_path, "w") as f:
    json.dump(agent_artifact, f, indent=2)

print(f"Saved stateful agent artifact to {save_path}")
assert os.path.exists(save_path), "Save failed!"
print("✅ Day 93 artifact saved.")

Saved stateful agent artifact to /content/drive/MyDrive/teachmewrite-ai/agent_stateful_day93.json
✅ Day 93 artifact saved.
